# Benim Sesimle Oku — Ücretsiz Türkçe Ses Klonlama
Bu defter Anka-TTS açık kaynak modelini kullanır. **Colab ücretsiz GPU erişimi garanti değildir.** Sesin bu Colab çalışma ortamına yüklenir. Ses örneği olarak yalnızca kendi sesini kullan.
**Önemli:** Referans kayıtta ne söylediğini birebir yazmalısın. Tercihen sessiz ortamda 7–9 saniyelik kayıt kullan.
Model tek seferde 7–9 saniyelik referans kullanır; beş kaydı otomatik eğitmez. Model ağırlıkları CC BY-NC 4.0 kapsamında, ticari kullanım için uygun değildir.


In [ ]:
!pip -q install 'anka-tts[tts]' soundfile
import torch
print('GPU:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Ücretsiz GPU oturumu bulunamadı. Colab > Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seçip yeniden dene.')

In [ ]:
from google.colab import files
print('7-9 saniyelik kendi sesini içeren MP3 veya WAV dosyasını yükle.')
uploaded = files.upload()
if not uploaded: raise RuntimeError('Ses yüklenmedi.')
reference = next(iter(uploaded))
print('Seçilen kayıt:',reference)

In [ ]:
#@title Kendi sesimle yeni metin oluştur
referans_metin = 'Ses kaydında söylediğin sözleri buraya birebir yaz.' #@param {type:'string'}
okunacak_metin = 'Merhaba, bu metni kendi sesimle okuyorum.' #@param {type:'string'}
if not referans_metin.strip() or not okunacak_metin.strip(): raise ValueError('İki metin de gerekli.')
import subprocess
subprocess.run(['ffmpeg','-y','-i',reference,'-ac','1','-ar','24000','referans.wav'],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
from anka import AnkaTTS
tts=AnkaTTS.from_pretrained('anka-tts/v0.1',device='cuda')
tts.add_voice('benim_sesim','referans.wav',referans_metin)
ses=tts.synthesize(okunacak_metin,voice='benim_sesim')
tts.save_wav(ses,'benim_sesim.wav')
from IPython.display import Audio,display
display(Audio('benim_sesim.wav'))
files.download('benim_sesim.wav')